In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import spatialdata as sd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import senepy as sp
from typing import Optional, Tuple, Literal
import math



In [ ]:
import sys
from pathlib import Path

# Locate project root (folder containing config.yaml) and add scripts/ to sys.path
_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})

# Tell matplotlib to export text as TrueType fonts for PDF and PS/EPS

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
tissue_colors = {'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}
tissue_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'AD': '#ffb142', 'CA': '#b33939'}

tissue_dysplasia_colors_normal_split = {'Dist_N': "#33fd55",'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}
tissue_dysplasia_colors = {'Adj_N': '#40407a', 'LGD':"#ffda79",'HGD':'#ff793f', 'CA': '#b33939'}

colors_dutch = ['#F79F1F',
 '#1289A7',
 '#009432',
 '#9980FA',
 '#EA2027',
 '#833471',
 '#1B1464']

colors_dutch_y = ['#F79F1F',
 '#1289A7',
 '#A3CB38',
 '#9980FA',
 '#ED4C67',
 '#833471',
 '#D980FA',
 '#009432',
 '#1B1464',
 '#0652DD',
'#EA2027',
 '#EE5A24']

colors_dutch_long = [
    '#FFC312', '#C4E538', '#12CBC4', '#FDA7DF', '#ED4C67',
    '#F79F1F', '#A3CB38', '#1289A7', '#D980FA', '#B53471',
    '#EE5A24', '#009432', '#0652DD', '#9980FA', '#833471',
    '#EA2027', '#006266', '#1B1464', '#5758BB', '#6F1E51'
]

# pre fibroblast filtering

In [ ]:

adjacent_normal_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_adjacent_normal_30_50.h5ad"))
separate_normal_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_distant_normal_30_50.h5ad"))
ta_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_AD_30_50_tt_harmony_batch_05_pt_05_ssg.h5ad"))
ca_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_CA_30_50_tt_harmony_batch_05_pt_05_ssg.h5ad"))

In [ ]:
tissue_adatas = {"adjacent_normal": (adjacent_normal_adata, "adjacent_normal_0.8", 5),
                 "separate_normal":(separate_normal_adata, "distant_normal_0.8",5),
                 "AD":(ta_adata,"epithelial_AD_0.8", 3),
                 "CA":(ca_adata,"epithelial_CA_0.8",5)}

In [ ]:
fig_outdir = str(P.results.figures / "supplemental-clustering")

In [ ]:
sc.settings._vector_friendly = True
os.makedirs(fig_outdir, exist_ok=True)

for key, value in tissue_adatas.items():
    adata = value[0]
    res = value[1]
    pt_size = value[2]

    # Set category order and store colors before any plotting
    groups = sorted(adata.obs[res].cat.categories, key=int)
    adata.obs[res] = adata.obs[res].cat.reorder_categories(groups)
    adata.uns[f"{res}_colors"] = [colors_dutch_y[i] for i, _ in enumerate(groups)]

    ax = sc.pl.umap(adata, color=res, show=False, size=pt_size, title=key)
    ax.set_axis_off()
    plt.savefig(os.path.join(fig_outdir, f"{key}_umap.pdf"), dpi=300, bbox_inches="tight")
    plt.show()

    sc.tl.rank_genes_groups(adata, res, layer='normalized', use_raw=False)
    # Delete any cached dendrogram so dotplot can't use it
    dend_key = f"dendrogram_{res}"
    if dend_key in adata.uns:
        del adata.uns[dend_key]

    sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, groupby=res,
                                    title=f"{res}", standard_scale="var",
                                    show=False, dendrogram=False)
    plt.savefig(os.path.join(fig_outdir, f"{key}_dotplot.pdf"), dpi=300, bbox_inches="tight")
    plt.show()

sc.settings._vector_friendly = False

# Normal post-fibroblast filtering:


In [ ]:
separate_normal_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_distant_normal_16_50.h5ad"))

In [ ]:
res = "distant_normal_filt_1.4"

In [ ]:
my_markers = {
        "Stem":['OLFM4', 'LGR5', 'SLC12A2','CD44'],
        "Transit Amplifying":['MKI67', 'PCNA', 'TOP2A', 'TUBB' ],
        "Goblet":["FCGBP","TFF3","ATOH1"],
        "Paneth":["DEFA6","TSPAN8"],
        "Tuft":["POU2F3","DCLK1", "SOX9"],
        "EE":["CHGA","NEUROD1","ENPP2"],
        "Colonocyte":["SLC26A3","AQP8","CEACAM1", "ANPEP","IL32","KRT20","SLC26A2","SELENBP1"]}

In [ ]:
sc.settings._vector_friendly = True

# Set category order and store colors
groups = sorted(separate_normal_adata.obs[res].cat.categories, key=int)
separate_normal_adata.obs[res] = separate_normal_adata.obs[res].cat.reorder_categories(groups)
separate_normal_adata.uns[f"{res}_colors"] = [colors_dutch_y[i] for i, _ in enumerate(groups)]

ax = sc.pl.umap(separate_normal_adata, color=res, show=False, size=5, title="Distant Normal")
ax.set_axis_off()
plt.savefig(os.path.join(fig_outdir, "distant_normal_filt_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

sc.tl.rank_genes_groups(separate_normal_adata, res, layer='normalized', use_raw=False)
dend_key = f"dendrogram_{res}"
if dend_key in separate_normal_adata.uns:
    del separate_normal_adata.uns[dend_key]

sc.pl.rank_genes_groups_dotplot(separate_normal_adata, n_genes=5, groupby=res,
                                title="Distant Normal", standard_scale="var",
                                show=False, dendrogram=False)
plt.savefig(os.path.join(fig_outdir, "distant_normal_filt_dotplot.pdf"), dpi=300, bbox_inches="tight")
plt.show()


sc.settings._vector_friendly = False

In [ ]:
rough_anno_dict = {
    "0": "Colonocytes",
    "1": "Colonocytes",
    "2": "Colonocyte Progenitor Cells",
"3": "Colonocytes",
"4": "Transit Amplifying Cells",
"5": "Enteroendocrine / Crypt Base Absorptive",
"6": "Stem Cells",
"7": "Goblet Cells",
"8": "Colonocytes",
"9": "BEST4 Cells",
"10": "Unknown"
}

In [ ]:
separate_normal_adata.obs['normal_epithelial_cell_type'] = separate_normal_adata.obs['distant_normal_filt_1.4'].astype(str).map(rough_anno_dict)


In [ ]:
colors_dutch_y = ['#833471',
 "#1AD1FF",
 '#A3CB38',
 '#F79F1F',
'#ED4C67',
 '#D980FA',
 '#0652DD',
 '#9980FA']

In [ ]:
sc.settings._vector_friendly = True

ax = sc.pl.umap(separate_normal_adata, color="normal_epithelial_cell_type",palette=colors_dutch_y, size=5, show=False)
ax.set_axis_off()
plt.savefig(os.path.join(fig_outdir, "distant_normal_filt_anno_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()


sc.settings._vector_friendly = False


In [ ]:
sep_n_color_dict = dict(zip(
    separate_normal_adata.obs["normal_epithelial_cell_type"].cat.categories,
    separate_normal_adata.uns["normal_epithelial_cell_type_colors"]
))

# adj normal post-fibroblast filtering

In [ ]:
adj_normal_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_adjacent_normal_16_50.h5ad"))


In [ ]:
res = "adjacent_normal_filt_0.5"
fig_outdir = str(P.results.figures / "supplemental-clustering")

In [ ]:
sc.settings._vector_friendly = True

# Set category order and store colors
groups = sorted(adj_normal_adata.obs[res].cat.categories, key=int)
adj_normal_adata.obs[res] = adj_normal_adata.obs[res].cat.reorder_categories(groups)
adj_normal_adata.uns[f"{res}_colors"] = [colors_dutch_y[i] for i, _ in enumerate(groups)]

ax = sc.pl.umap(adj_normal_adata, color=res, show=False, size=5, title="Adjacent Normal")
ax.set_axis_off()
plt.savefig(os.path.join(fig_outdir, "adj_normal_filt_umap.pdf"), dpi=300, bbox_inches="tight")
plt.show()

sc.tl.rank_genes_groups(adj_normal_adata, res, layer='normalized', use_raw=False)
dend_key = f"dendrogram_{res}"
if dend_key in adj_normal_adata.uns:
    del adj_normal_adata.uns[dend_key]

sc.pl.rank_genes_groups_dotplot(adj_normal_adata, n_genes=5, groupby=res,
                                title="Adjacent Normal", standard_scale="var",
                                show=False, dendrogram=False)
plt.savefig(os.path.join(fig_outdir, "adj_normal_filt_dotplot.pdf"), dpi=300, bbox_inches="tight")
plt.show()




sc.settings._vector_friendly = False